In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

mkt = pd.read_csv(
    r'C:\Users\user\DSS_Project\data\clean\marketing_clean.csv')

print(f"✅ Marketing loaded: {mkt.shape}")

✅ Marketing loaded: (60000, 23)


In [2]:
df = mkt.copy()

# Target — Revenue above median
df['success'] = (
    df['sales_revenue_usd'] > df['sales_revenue_usd'].median()
).astype(int)

# New powerful features
df['roi'] = (
    df['sales_revenue_usd'] / 
    df['marketing_budget_usd'].replace(0, np.nan)
).fillna(0).round(4)

df['online_ratio'] = (
    df['ad_spend_online_usd'] / 
    df['marketing_budget_usd'].replace(0, np.nan)
).fillna(0).round(4)

df['revenue_per_traffic'] = (
    df['sales_revenue_usd'] / 
    df['website_traffic'].replace(0, np.nan)
).fillna(0).round(4)

df['budget_efficiency'] = (
    df['sales_revenue_usd'] / 
    df['marketing_budget_usd'].replace(0, np.nan)
).fillna(0).round(4)

df['promo_effectiveness'] = (
    df['sales_revenue_usd'] / 
    df['num_promotions'].replace(0, np.nan)
).fillna(0).round(4)

df['email_revenue'] = (
    df['email_open_rate'] * 
    df['conversion_rate'] * 
    df['sales_revenue_usd']
).round(4)

df['social_revenue'] = (
    df['social_media_followers'] * 
    df['conversion_rate']
).round(4)

df['spend_ratio'] = (
    df['ad_spend_online_usd'] / 
    df['ad_spend_offline_usd'].replace(0, np.nan)
).fillna(0).round(4)

df['engagement_score'] = (
    df['email_open_rate'] * 
    df['social_media_followers'] * 
    df['conversion_rate']
).round(4)

df['customer_value'] = (
    df['num_previous_purchases'] * 
    df['customer_satisfaction_score']
).round(4)

# Encode categoricals
le = LabelEncoder()
for col in ['sales_channel', 'product_category', 
            'customer_segment', 'region', 'season']:
    df[col+'_enc'] = le.fit_transform(df[col].astype(str))

print(f"✅ Features after engineering: {df.shape[1]}")
print(f"✅ Success Rate: {df['success'].mean()*100:.1f}%")

✅ Features after engineering: 39
✅ Success Rate: 50.0%


In [3]:
features = [
    # Original features
    'marketing_budget_usd', 'ad_spend_online_usd',
    'ad_spend_offline_usd', 'num_promotions',
    'discount_percentage', 'num_sales_representatives',
    'customer_age', 'customer_satisfaction_score',
    'competitor_price_index', 'website_traffic',
    'conversion_rate', 'email_open_rate',
    'social_media_followers', 'days_since_last_purchase',
    'num_previous_purchases',
    # Encoded
    'sales_channel_enc', 'product_category_enc',
    'customer_segment_enc', 'region_enc', 'season_enc',
    # New features
    'roi', 'online_ratio', 'revenue_per_traffic',
    'budget_efficiency', 'promo_effectiveness',
    'email_revenue', 'social_revenue', 'spend_ratio',
    'engagement_score', 'customer_value'
]

X = df[features].fillna(0)
y = df['success']

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# SMOTE
X_bal, y_bal = SMOTE(random_state=42).fit_resample(X_scaled, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42)

print(f"✅ Training size : {X_train.shape[0]:,}")
print(f"✅ Testing size  : {X_test.shape[0]:,}")
print(f"✅ Features      : {X.shape[1]}")

✅ Training size : 48,000
✅ Testing size  : 12,000
✅ Features      : 30


In [4]:
model_xgb = XGBClassifier(
    n_estimators=700,
    learning_rate=0.01,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=1,
    eval_metric='logloss'
)

model_xgb.fit(X_train, y_train)
y_pred_xgb = model_xgb.predict(X_test)

print("✅ XGBoost Improved:")
print(classification_report(y_test, y_pred_xgb))

✅ XGBoost Improved:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6028
           1       1.00      1.00      1.00      5972

    accuracy                           1.00     12000
   macro avg       1.00      1.00      1.00     12000
weighted avg       1.00      1.00      1.00     12000



In [5]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

estimators = [
    ('xgb', XGBClassifier(
        n_estimators=500, learning_rate=0.02,
        max_depth=6, random_state=42,
        eval_metric='logloss', n_jobs=1)),
    ('rf', RandomForestClassifier(
        n_estimators=300, max_depth=8,
        random_state=42, n_jobs=1)),
    ('gb', GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.03,
        max_depth=5, random_state=42))
]

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=1
)

stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)

print("✅ Stacking Improved:")
print(classification_report(y_test, y_pred_stack))

✅ Stacking Improved:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6028
           1       1.00      1.00      1.00      5972

    accuracy                           1.00     12000
   macro avg       1.00      1.00      1.00     12000
weighted avg       1.00      1.00      1.00     12000



In [6]:
import sqlite3
import os

# Best predictions
best_pred = stack_model.predict_proba(X_test)[:, 1]

print("="*50)
print("FINAL COMPARISON")
print("="*50)
print(f"Baseline GradientBoosting : 68.44%")
print(f"XGBoost Improved          : {(y_pred_xgb == y_test).mean()*100:.2f}%")
print(f"Stacking Improved         : {(y_pred_stack == y_test).mean()*100:.2f}%")

# Save results
results_df = pd.DataFrame({
    'actual': y_test,
    'predicted': y_pred_stack,
    'probability': best_pred
})

os.makedirs(
    r'C:\Users\user\DSS_Project\exports', exist_ok=True)

results_df.to_csv(
    r'C:\Users\user\DSS_Project\exports\marketing_model_improved.csv',
    index=False)

results_df.to_csv(
    r'C:\Users\user\DSS_Project\exports\powerbi\marketing_model_powerbi.csv',
    index=False)

print("\n✅ Results saved successfully")

FINAL COMPARISON
Baseline GradientBoosting : 68.44%
XGBoost Improved          : 99.59%
Stacking Improved         : 99.76%

✅ Results saved successfully
